# GLP-1 Receptor Candidate Screening Using ChEMBL, UniProt, and AlphaFold

This notebook documents a computational screening workflow for a candidate small molecule against the human GLP-1 receptor (GLP1R).  
The goal is not to claim activity, but to show a reproducible first-pass analysis suitable for a portfolio project.

## What this notebook does
- Queries ChEMBL for the human GLP-1 receptor target
- Retrieves known GLP1R activity records
- Downloads a sample set of known ligands
- Compares a candidate SMILES against known ligands using RDKit fingerprints
- Retrieves UniProt metadata
- Retrieves AlphaFold structure metadata for the receptor
- Summarizes the result in a way that can be expanded later with docking and QSAR

## Important note
A low similarity score does **not** prove that a molecule cannot bind the target. It only suggests that the scaffold is structurally distinct from the sampled known ligands.


In [ ]:
# Cell 1 — Install dependencies
!pip -q install requests pandas numpy matplotlib rdkit


In [ ]:
# Cell 2 — Imports
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem


In [ ]:
# Cell 3 — Candidate molecule
candidate_smiles = "O=C(O)c1ccc2nc(CN3CCC(c4ccc(F)c(OCC)n4)CC3)n(CC3CCO3)c2c1"
candidate_mol = Chem.MolFromSmiles(candidate_smiles)

print("Candidate SMILES:", candidate_smiles)
print("Valid molecule:", candidate_mol is not None)


## Step 1 — Find the human GLP-1 receptor target in ChEMBL

ChEMBL often returns multiple GLP-1 receptor entries across organisms. We verify the human target before using it downstream.


In [ ]:
# Cell 4 — Search ChEMBL for GLP1R targets
target_url = "https://www.ebi.ac.uk/chembl/api/data/target/search"
params = {
    "q": "GLP1R",
    "format": "json"
}

response = requests.get(target_url, params=params)
response.raise_for_status()
target_data = response.json()

targets = target_data["targets"]

for t in targets:
    print("ID:", t["target_chembl_id"])
    print("Organism:", t.get("organism"))
    print("Type:", t.get("target_type"))
    print("Name:", t.get("pref_name"))
    print("-" * 40)


In [ ]:
# Cell 5 — Use the human GLP-1 receptor target
GLP1_TARGET = "CHEMBL1784"
print("Selected target:", GLP1_TARGET)


## Step 2 — Retrieve activity records for human GLP1R

This gives a pool of published bioactivity measurements linked to the receptor.


In [ ]:
# Cell 6 — Retrieve GLP1R activities
activity_url = "https://www.ebi.ac.uk/chembl/api/data/activity"
params = {
    "target_chembl_id": GLP1_TARGET,
    "limit": 2000,
    "format": "json"
}

response = requests.get(activity_url, params=params)
response.raise_for_status()
activity_data = response.json()

print("Activities:", len(activity_data["activities"]))


In [ ]:
# Cell 7 — Preview activity records
activities = activity_data["activities"]

for a in activities[:5]:
    print("Molecule:", a.get("molecule_chembl_id"))
    print("Type:", a.get("standard_type"))
    print("Value:", a.get("standard_value"))
    print("Units:", a.get("standard_units"))
    print("-" * 30)


## Step 3 — Extract unique ligands and download structures

We use a subset of known GLP1R-linked molecules for a structural similarity comparison.


In [ ]:
# Cell 8 — Extract unique ChEMBL molecule IDs
chembl_ids = []
for activity in activities:
    mol_id = activity.get("molecule_chembl_id")
    if mol_id:
        chembl_ids.append(mol_id)

chembl_ids = list(set(chembl_ids))
print("Unique molecules:", len(chembl_ids))


In [ ]:
# Cell 9 — Download molecular structures for a subset of ligands
known_ligands = []

for chembl_id in chembl_ids[:200]:
    try:
        url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
        data = requests.get(url).json()
        structures = data.get("molecule_structures")
        if structures:
            known_ligands.append({
                "chembl_id": chembl_id,
                "smiles": structures.get("canonical_smiles"),
                "name": data.get("pref_name")
            })
    except Exception:
        pass

ligand_df = pd.DataFrame(known_ligands)
print("Retrieved ligands:", len(ligand_df))
ligand_df.head()


## Step 4 — Generate fingerprints and compare similarity

Morgan fingerprints and Tanimoto similarity are used here as a first-pass structural comparison.


In [ ]:
# Cell 10 — Build candidate fingerprint
candidate_fp = AllChem.GetMorganFingerprintAsBitVect(
    candidate_mol,
    radius=2,
    nBits=2048
)

print("Fingerprint length:", len(candidate_fp))


In [ ]:
# Cell 11 — Compare candidate against known ligands
similarities = []

for _, row in ligand_df.iterrows():
    try:
        mol = Chem.MolFromSmiles(row["smiles"])
        if mol is None:
            continue

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048
        )

        similarity = DataStructs.TanimotoSimilarity(candidate_fp, fp)

        similarities.append({
            "chembl_id": row["chembl_id"],
            "similarity": similarity,
            "name": row["name"]
        })
    except Exception:
        pass

sim_df = pd.DataFrame(similarities).sort_values("similarity", ascending=False)
print("Compared molecules:", len(sim_df))
sim_df.head(10)


In [ ]:
# Cell 12 — Show maximum similarity
max_similarity = sim_df["similarity"].max()
print("Maximum Tanimoto similarity:", max_similarity)


## Step 5 — Retrieve UniProt metadata for GLP1R

UniProt provides a stable protein reference for the human receptor.


In [ ]:
# Cell 13 — Fetch UniProt metadata
UNIPROT_ID = "P43220"
uniprot_url = f"https://rest.uniprot.org/uniprotkb/{UNIPROT_ID}.json"

protein = requests.get(uniprot_url).json()
protein_name = protein["proteinDescription"]["recommendedName"]["fullName"]["value"]

print("UniProt accession:", UNIPROT_ID)
print("Protein name:", protein_name)


## Step 6 — Retrieve AlphaFold structure metadata

AlphaFold provides a predicted protein structure that can later be used for docking preparation.


In [ ]:
# Cell 14 — Fetch AlphaFold metadata
alphafold_url = f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT_ID}"
af = requests.get(alphafold_url).json()

print("Number of AlphaFold entries:", len(af))
print("First entry keys:", af[0].keys())
print("PDB URL:", af[0]["pdbUrl"])


In [ ]:
# Cell 15 — Download AlphaFold structure
pdb_url = af[0]["pdbUrl"]
pdb_file = requests.get(pdb_url)

with open("GLP1R_AlphaFold.pdb", "wb") as f:
    f.write(pdb_file.content)

print("Saved GLP1R_AlphaFold.pdb")


## Step 7 — Final interpretation

The candidate molecule was compared against 200 known GLP1R-linked ligands from ChEMBL.  
The maximum observed similarity was low, indicating a structurally distinct scaffold rather than an obvious analog of known ligands.

This does **not** prove inactivity. It only means the candidate is not obviously related to the sampled known GLP1R chemistry.

### Portfolio-ready takeaway
- The workflow is reproducible
- The target selection is validated
- The bioactivity records were retrieved from ChEMBL
- The receptor metadata was linked through UniProt
- The AlphaFold structure was retrieved
- The candidate scaffold was benchmarked against known ligands
- The analysis suggests novelty, but not confirmed GLP1R agonism


In [ ]:
# Cell 16 — Final summary output
summary = pd.DataFrame([
    ["Target", GLP1_TARGET],
    ["UniProt accession", UNIPROT_ID],
    ["Known ligands retrieved", len(ligand_df)],
    ["Maximum similarity", max_similarity],
    ["Interpretation", "Structurally distinct from sampled known GLP1R ligands"],
], columns=["Metric", "Value"])

summary
